# 02 探索性分析

按客户分层、渠道、品类、设备、地区逐一拆解购买率差异。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')
from utils.data_loader import load_cleaned_data

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

df = load_cleaned_data()
print(f'{len(df):,} customers')

In [ ]:
# 客户分层的购买率
seg_rate = df.groupby('CustomerSegment')['PurchaseStatus'].agg(['mean','count'])
seg_rate['mean'] = seg_rate['mean'] * 100
print('CustomerSegment purchase rate:')
print(seg_rate.to_string())
print('\n注意：三个分层的所有行为指标（购买次数、收入、满意度、停留时间）几乎完全相同——')
print('标签不反映实际行为差异。这不是"VIP 不如 Regular"，而是"标签定义需要追问"。')
print('面试中这是一个展示分析纪律的机会：先验证标签，再使用标签。')

In [ ]:
# 忠诚度计划效果
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

loyalty = df.groupby('LoyaltyProgram')['PurchaseStatus'].mean() * 100
axes[0].bar(['Non-Member','Member'], [loyalty[0], loyalty[1]], color=['#C44E52','#4C72B0'])
for i, v in enumerate([loyalty[0], loyalty[1]]):
    axes[0].text(i, v+1, f'{v:.1f}%', ha='center', fontsize=14, fontweight='bold')
axes[0].set_title(f'Loyalty Program: +{loyalty[1]-loyalty[0]:.1f}pp Lift', fontsize=13, fontweight='bold')

# 品类对比
cat_rate = df.groupby('ProductCategory')['PurchaseStatus'].mean().sort_values() * 100
axes[1].barh(range(len(cat_rate)), cat_rate.values, color=['#4C72B0','#55A868','#C44E52','#DD8452','#937860'])
axes[1].set_yticks(range(len(cat_rate)))
axes[1].set_yticklabels(cat_rate.index)
axes[1].set_title('Purchase Rate by Category', fontsize=13, fontweight='bold')
for i, v in enumerate(cat_rate.values):
    axes[1].text(v+0.2, i, f'{v:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()
print(f'品类差异仅 {cat_rate.max()-cat_rate.min():.1f}pp — 品类不是购买决策的主要驱动因素')

In [ ]:
# 渠道 ROI
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ref_rate = df.groupby('ReferralSource')['PurchaseStatus'].mean().sort_values() * 100
axes[0].barh(range(len(ref_rate)), ref_rate.values, color=['#4C72B0','#55A868','#C44E52','#DD8452','#937860'])
axes[0].set_yticks(range(len(ref_rate)))
axes[0].set_yticklabels(ref_rate.index)
axes[0].set_title('Purchase Rate by Channel', fontsize=13, fontweight='bold')
for i, v in enumerate(ref_rate.values):
    axes[0].text(v+0.2, i, f'{v:.1f}%', va='center', fontsize=10)

# 渠道分布 vs 转化
ref_summary = df.groupby('ReferralSource').agg(volume=('PurchaseStatus','count'),rate=('PurchaseStatus','mean'))
ref_summary['rate'] = ref_summary['rate'] * 100
axes[1].scatter(ref_summary['volume'], ref_summary['rate'], s=ref_summary['volume']/1000, alpha=0.6)
for idx, row in ref_summary.iterrows():
    axes[1].annotate(idx, (row['volume'], row['rate']), fontsize=10)
axes[1].set_xlabel('Traffic Volume', fontsize=11)
axes[1].set_ylabel('Purchase Rate (%)', fontsize=11)
axes[1].set_title('Channel Volume vs Conversion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'渠道差异仅 {ref_rate.max()-ref_rate.min():.1f}pp — 所有渠道转化率极其接近')

In [ ]:
# 满意度分布
fig, ax = plt.subplots(figsize=(10, 5))
for status, label, color in [(1,'Purchased','#4C72B0'),(0,'Not Purchased','#C44E52')]:
    sdf = df[df['PurchaseStatus'] == status]
    sat_dist = sdf['CustomerSatisfaction'].value_counts().sort_index()
    ax.plot(sat_dist.index, sat_dist.values, 'o-', label=label, color=color, markersize=10, linewidth=2)
ax.set_xlabel('Satisfaction Score', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Satisfaction Distribution by Purchase Status', fontsize=13, fontweight='bold')
ax.legend()
ax.set_xticks([1,2,3,4,5])
plt.tight_layout()
plt.show()
print('满意度 4-5 分的用户购买率明显高于 1-2 分——体验驱动的转化')

## 探索小结

- 预置标签（VIP/Premium/Regular）不反映购买行为差异——三个分层的所有指标几乎完全相同，标签定义需追问数据来源
- 忠诚度计划带来 8.4pp 的购买率提升 — **最大的单一业务杠杆**
- 品类、渠道、设备、地区差异均在 2pp 以内 — 人口统计和渠道不是核心驱动力
- 满意度越高购买率越高 — 体验质量驱动转化

下一步 → `03_purchase_drivers.ipynb`